In [3]:
import warnings
warnings.filterwarnings("ignore")


Question 1:

Task: Implement a FAISS-based Retrieval System that fetches the most relevant document for query-based retrieval.

In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Sample document corpus
documents = [
    "Python is a popular programming language.",
    "FAISS is used for efficient similarity search.",
    "Generative AI creates new content using models.",
    "Machine learning enables computers to learn from data."
]

# 2. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Convert documents to embeddings
doc_embeddings = model.encode(documents)

# 4. Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print("FAISS index ready.")

# 5. Query function
def retrieve(query, k=1):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [documents[i] for i in indices[0]]

# 6. Test query
query = "What is FAISS used for?"
results = retrieve(query)

print("\nQuery:", query)
print("Top result:", results[0])


FAISS index ready.

Query: What is FAISS used for?
Top result: FAISS is used for efficient similarity search.


Question 2
Task: Use Gemini API to generate answers based on retrieved knowledge

In [7]:
import os
from google import genai

# Configure API key (make sure GEMINI_API_KEY is set)
client = genai.Client(api_key=API_Key)

# Example retrieved document (from your FAISS system)
retrieved_doc = "FAISS is used for efficient similarity search in large datasets."

query = "What is FAISS used for?"

# Build prompt
prompt = f"""
Use the following context to answer the question.

Context:
{retrieved_doc}

Question:
{query}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)


FAISS is used for efficient similarity search in large datasets.


Question 3
Task: Implement FAISS with Hybrid (Dense + Sparse) Search

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample documents
documents = [
    "FAISS is used for similarity search.",
    "Python is a programming language.",
    "Transformers are used in NLP."
]

# Dense embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
dense_embeddings = model.encode(documents)

# FAISS index
dimension = dense_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(dense_embeddings))

# Sparse (TF-IDF)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Query
query = "What is FAISS?"

# Dense search
query_dense = model.encode([query])
_, dense_idx = index.search(np.array(query_dense), k=1)

# Sparse search
query_sparse = vectorizer.transform([query])
sparse_scores = (tfidf_matrix @ query_sparse.T).toarray().flatten()
sparse_idx = sparse_scores.argmax()

# Hybrid decision (simple vote)
final_idx = dense_idx[0][0]

print("Top document:", documents[final_idx])


Top document: FAISS is used for similarity search.


Question 4
Task: Multi-Turn Conversational Chatbot using Gemini

In [11]:
import os
from google import genai

client = genai.Client(api_key=API_Key)

chat_history = []

def chat(user_input):
    chat_history.append(f"User: {user_input}")

    prompt = "\n".join(chat_history) + "\nAssistant:"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text
    chat_history.append(f"Assistant: {answer}")

    return answer

# ---- conversation ----
print(chat("What is FAISS?"))
print(chat("Why is it useful?"))


**FAISS** stands for **Facebook AI Similarity Search**.

It's an open-source library developed by Facebook AI Research (FAIR) for **efficient similarity search** and **clustering of dense vectors**. In simpler terms, it's designed to help you quickly find the most similar items to a given query when those items are represented as high-dimensional vectors (often called "embeddings").

Here's a breakdown of what that means and why it's so important:

1.  **High-Dimensional Vectors (Embeddings):** In modern AI, complex data like images, text, audio, or user preferences are often converted into numerical lists called vectors. These vectors capture the "meaning" or "features" of the data, and similar items will have vectors that are "close" to each other in a multi-dimensional space. "High-dimensional" means these vectors can have hundreds or even thousands of numbers.

2.  **Similarity Search (Nearest Neighbors):** The goal is to find items whose vectors are closest to a given query vector

Question 5
Task: FAISS-based RAG system using Gemini

In [13]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from google import genai


client = genai.Client(api_key=API_Key)

documents = [
    "FAISS is used for efficient similarity search.",
    "Python is popular for machine learning.",
    "RAG combines retrieval with generation."
]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready.")

# RAG function
def rag_answer(query):

    # Retrieve
    query_vec = model.encode([query])
    _, idx = index.search(np.array(query_vec), k=1)
    context = documents[idx[0][0]]

    # Generate with Gemini
    prompt = f"""
    Use the context to answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return context, response.text


# Test
query = "What is FAISS used for?"
context, answer = rag_answer(query)

print("Retrieved:", context)
print("Answer:", answer)


FAISS index ready.
Retrieved: FAISS is used for efficient similarity search.
Answer: FAISS is used for efficient similarity search.
